# Seleção e pré-processamento — prêmios validados
Configure `BASE_PREMIOS` na primeira célula de configuração para apontar ao `categorias_ic/classificacao_detalhada.csv` gerado pelo script de consolidação. O notebook preserva os arquivos de entrada e grava os resultados em `resultado_analise_validada/`. Os prêmios são lidos diretamente da base oficial, sem reclassificação por palavras-chave.


In [1]:
import pandas as pd
from collections import Counter, defaultdict, OrderedDict
import csv
import re
import unicodedata
from pathlib import Path


In [7]:
# Configuração: ajuste somente os caminhos, se necessário.
PASTA_ENTRADA = Path('.')
BASE_PREMIOS = Path('classificacao_detalhada.csv')
# Se a base estiver junto do notebook, ajuste para:
# BASE_PREMIOS = PASTA_ENTRADA / 'classificacao_detalhada.csv'
PASTA_SAIDA = Path('resultado_analise_validada')
PASTA_SAIDA.mkdir(parents=True, exist_ok=True)

def entrada(nome, alternativa=None):
    caminho = PASTA_ENTRADA / nome
    if not caminho.is_file() and alternativa:
        caminho = PASTA_ENTRADA / alternativa
    if not caminho.is_file():
        raise FileNotFoundError(caminho)
    return caminho

def saida(nome):
    return PASTA_SAIDA / nome

# Lendo todos os arquivos

# Anuário
anuario = entrada("Anuário 2015-2025.xlsx", "Anuário 2015-2025(5).xlsx")
abas = pd.read_excel(anuario, sheet_name=None)
df_docentes = abas["Docentes Ativos "]
df_magisterio = abas["Magistério"]
df_sexo = abas["Sexo"]
df_raca = abas["Raça"]

# Bancas
anos = [2023, 2024, 2025]

lista_dfs = []

for ano in anos:
    arquivo = entrada(f"Classificação_{ano}_premios_validados.xlsx",
                      f"Classificação_{ano}_premios_validados(2).xlsx")

    df = pd.read_excel(arquivo, sheet_name=0)

    df["Ano"] = ano

    lista_dfs.append(df)

df_unificado = pd.concat(lista_dfs, ignore_index=True)

df_unificado.to_excel(
    saida("Classificação_2023_2025_unificada.xlsx"),
    index=False
)

# Relação gênero e raça dos professores
gen_raça = entrada("Relação gênero e raça IC.xlsx", "Relação gênero e raça IC(2).xlsx")
df_gen = pd.read_excel(gen_raça, sheet_name=0)
print("Cadastro nominal carregado:", len(df_gen), "linhas")

Cadastro nominal carregado: 85 linhas


In [8]:
# Funções de padronização de grafias e leituras de linhas

# Os números de linha começam em 2, porque o cabeçalho ocupa a primeira linha do Excel.
def converter_linhas(df):
    linhas = []
    for indice, valores in enumerate(df.itertuples(index=False, name=None), start=2):
        valores = [None if pd.isna(v) else v for v in valores]
        linhas.append((indice, valores))
    return linhas

def limpar(valor):
    return ' '.join(str(valor).split()) if valor is not None else ''

def normalizar(valor):
    sem_acento = unicodedata.normalize('NFKD', limpar(valor).casefold())
    return ' '.join(''.join(c for c in sem_acento if not unicodedata.combining(c)).split()).replace('’', "'")

def celula(linha, i):
    return linha[i] if i < len(linha) else None

# Correções manuais de grafias equivalentes
ALIAS = {'ruben interian kovaliova': 'ruben interian'}
def chave(nome):
    t = normalizar(nome)
    return ALIAS.get(t, t)

cadastro = {}
for linha_excel, campos in converter_linhas(df_gen):
    nome = limpar(celula(campos, 0))
    if not nome:
        continue
    cadastro[chave(nome)] = {
        'nome': nome,
        'cargo': limpar(celula(campos, 1)),
        'ms': limpar(celula(campos, 2)),
        'genero': limpar(celula(campos, 3)),
        'raca_cor': limpar(celula(campos, 4)),
        'origem_genero_raca': limpar(celula(campos, 5)),
        'situacao': limpar(celula(campos, 6)),
        'linha_cadastro': linha_excel,
    }
print('Nomes na relação de gênero/raça:', len(cadastro))



Nomes na relação de gênero/raça: 85


In [9]:
# Identificar defesas e funções dos docentes
defesas_xlsx = [(1, df_unificado.columns.tolist())] + converter_linhas(df_unificado)
cabecalho = defesas_xlsx[0][1]
assert len(cabecalho) == 53, 'Confira se a estrutura da classificação unificada mudou.'

FUNCOES = [(str(cabecalho[i]), i, i+1, i+2) for i in (2,7,12,17,22,27,32,37,42,47)]

defesa_validas = []
linhas_descartadas = []
for linha_excel, linha in defesas_xlsx[1:]:
    nivel = normalizar(celula(linha, 0))
    estudante = limpar(celula(linha, 1))
    if nivel not in ('mestrado', 'doutorado') or not estudante:
        linhas_descartadas.append(linha_excel)
        continue
    ano = int(celula(linha, 52))
    defesa_validas.append((linha_excel, linha, ano))

# Nome explicitamente associado a IC/UNICAMP em qualquer defesa.
nomes_ic_explicitos = Counter()
grafias_ic = defaultdict(Counter)
for linha_excel, linha, ano in defesa_validas:
    for funcao, i_nome, i_inst, i_premio in FUNCOES:
        nome = limpar(celula(linha, i_nome))
        instituto = normalizar(celula(linha, i_inst)).replace(' ', '')
        if nome and instituto == 'ic/unicamp':
            nomes_ic_explicitos[chave(nome)] += 1
            grafias_ic[chave(nome)][nome] += 1

print('Defesas válidas:', len(defesa_validas))
print('Funções no arquivo:', [x[0] for x in FUNCOES])

Defesas válidas: 173
Funções no arquivo: ['Orientador', 'Co-orientador', 'Titular 1', 'Titular 2', 'Titular 3', 'Titular 4', 'Titular 5', 'Suplente 1', 'Suplente 2', 'Suplente 3']


In [10]:
# Pegando as participações dos docentes com vínculo com IC
# Caso peculiar da planilha: uma célula contém duas pessoas.
# A primeira é conhecida do IC, e a segunda requer análise separada.
NOMES_CONJUNTOS = {
    normalizar('Alexandre Xavier Falcão e Jean Cousty'): ['Alexandre Xavier Falcão'],
    normalizar('Leonardo Montecchi e Julio Cesar dos Reis'): ['Leonardo Montecchi', 'Julio Cesar dos Reis'],
}

def verificar_vinculo(nome, instituto):
    cod = chave(nome)
    uni = normalizar(instituto).replace(' ', '')
    if uni == 'ic/unicamp':
        return True, 'IC explícito'
    if not uni and cod in cadastro:
        return True, 'IC inferido pelo cadastro'
    if not uni and cod in nomes_ic_explicitos:
        return True, 'IC inferido por outra defesa'
    if uni in ('unicamp', 'universidadeestadualdecampinas') and cod in cadastro:
        return True, 'UNICAMP sem unidade; IC pelo cadastro'
    return False, 'Vínculo IC não comprovado'

participacoes = []
pendencias = []
for linha_excel, linha, ano in defesa_validas:
    nivel = limpar(celula(linha, 0))
    aluno = limpar(celula(linha, 1))
    for funcao, i_nome, i_inst, i_premio in FUNCOES:
        nome_original = limpar(celula(linha, i_nome))
        instituto = limpar(celula(linha, i_inst))
        premios_brutos = limpar(celula(linha, i_premio))
        if not nome_original or normalizar(nome_original) in ('nao ha','n/a','nenhum','sem suplentes','-'):
            continue
        partes = NOMES_CONJUNTOS.get(normalizar(nome_original), [nome_original])
        if partes != [nome_original]:
            pendencias.append([ano, aluno, nome_original, funcao, instituto, linha_excel,
                                'Nome conjunto: analisados os nomes do IC separadamente; célula original preservada'])
        for nome in partes:
            incluir, justificativa = verificar_vinculo(nome, instituto)
            if not incluir:
                if not instituto:
                    pendencias.append([ano, aluno, nome, funcao, instituto, linha_excel,
                                        'Instituto vazio sem confirmação de vínculo IC'])
                continue
            cod = chave(nome)
            nome_padrao = (cadastro[cod]['nome'] if cod in cadastro
                           else grafias_ic[cod].most_common(1)[0][0] if grafias_ic[cod]
                           else nome)
            participacoes.append({
                'chave': cod, 'nome': nome_padrao, 'ano': ano, 'nivel': nivel,
                'aluno': aluno, 'funcao': funcao, 'instituto': instituto,
                'criterio_vinculo': justificativa, 'linha_excel': linha_excel,
                'premios_brutos': premios_brutos,
            })
            if 'inferido' in justificativa or 'sem unidade' in justificativa:
                pendencias.append([ano, aluno, nome, funcao, instituto, linha_excel,
                                    'Incluído com vínculo inferido: conferir'])

# Uma participação = uma combinação de docente + defesa + função.
ids = [(p['chave'],p['linha_excel'],p['funcao']) for p in participacoes]
assert len(ids) == len(set(ids)), 'Há funções repetidas para a mesma pessoa/defesa.'
print('Participações IC:',len(participacoes))

Participações IC: 725


In [11]:
# Registro único de prêmios — fonte oficial validada, não as células das bancas.
# Não usa regex para classificar prêmios nem exige presença em uma banca.
if not BASE_PREMIOS.is_file():
    raise FileNotFoundError(f'Base oficial não encontrada: {BASE_PREMIOS}. Ajuste BASE_PREMIOS no início do notebook.')
base_premios = pd.read_csv(BASE_PREMIOS, sep=';', encoding='utf-8-sig', dtype=str, keep_default_na=False)
obrigatorias = {'Ano', 'Professor / nome', 'Prêmio', 'Categoria', 'Situação da premiação'}
if not obrigatorias.issubset(base_premios.columns):
    raise ValueError(f'Colunas faltantes na base: {obrigatorias - set(base_premios.columns)}. Use classificacao_detalhada.csv.')
MAPA_SITUACAO = {
    'Premiação / colocação': 'Prêmio/menção informada',
    'Menção honrosa': 'Prêmio/menção informada',
    'Finalista / indicação': 'Finalista/indicação',
    'Outro reconhecimento': 'Outros reconhecimentos',
}
linhas_por_prof = defaultdict(set)
for p in participacoes:
    linhas_por_prof[p['chave']].add(p['linha_excel'])
premios_unicos = OrderedDict()
for r in base_premios.to_dict('records'):
    nome = limpar(r['Professor / nome'])
    cod = chave(nome)
    ano_premio = int(r['Ano'])
    titulo = r['Prêmio']
    situacao = r['Situação da premiação']
    categoria = r['Categoria']
    if not cod or not titulo.strip() or not categoria.strip() or situacao not in MAPA_SITUACAO:
        raise ValueError(f'Registro oficial incompleto ou situação desconhecida: {nome}, {ano_premio}, {situacao}')
    id_premio = (cod, ano_premio, normalizar(titulo))
    if id_premio in premios_unicos:
        raise ValueError(f'Chave duplicada na base oficial: {id_premio}. Confira a fonte, sem descartar automaticamente.')
    premios_unicos[id_premio] = {
        'chave': cod, 'nome': nome, 'ano': ano_premio, 'titulo': titulo,
        'categoria': categoria, 'situacao': situacao, 'tipo': MAPA_SITUACAO[situacao],
        # Linhas de participação do docente, não fontes originais do prêmio.
        'linhas_fonte': linhas_por_prof[cod].copy(),
    }
# Impede que o filtro final descarte um docente premiado silenciosamente.
sem_cadastro = sorted({p['nome'] for p in premios_unicos.values()
                      if p['chave'] not in cadastro or cadastro[p['chave']]['situacao'] == 'Verificar vínculo'})
if sem_cadastro:
    raise ValueError(f'Docentes da base oficial ausentes ou pendentes no cadastro: {sem_cadastro}. Revise o cadastro.')
assert len(premios_unicos) == len(base_premios)
print('Registros oficiais de prêmios:', len(premios_unicos))
print('Docentes premiados:', len({p['chave'] for p in premios_unicos.values()}))
print('Categorias originais:', base_premios['Categoria'].nunique())
print(base_premios['Situação da premiação'].value_counts().to_string())
# Exporta todos os campos originais para rastreabilidade, sem reclassificar.
base_premios.to_csv(saida('premios_docentes_detalhados.csv'), sep=';', index=False, encoding='utf-8-sig')
resumo_premios_categoria = pd.crosstab(base_premios['Categoria'], base_premios['Situação da premiação'])
resumo_premios_categoria = resumo_premios_categoria.reindex(columns=list(MAPA_SITUACAO), fill_value=0)
resumo_premios_categoria['Total'] = resumo_premios_categoria.sum(axis=1)
resumo_premios_categoria.to_csv(saida('resumo_premios_por_categoria.csv'), sep=';', encoding='utf-8-sig')


Registros oficiais de prêmios: 293
Docentes premiados: 44
Categorias originais: 28
Situação da premiação
Premiação / colocação    213
Menção honrosa            36
Outro reconhecimento      28
Finalista / indicação     16


In [12]:
# Contabilizando por docente
pessoas = {cod: dados.copy() for cod, dados in cadastro.items()}
for p in participacoes:
    if p['chave'] not in pessoas:
        pessoas[p['chave']] = {
            'nome': p['nome'], 'cargo': 'Não informado', 'ms': '',
            'genero': 'Não informado', 'raca_cor': 'Não informado',
            'origem_genero_raca': 'Ausente da relação nominal',
            'situacao': 'Verificar vínculo', 'linha_cadastro': '',
        }

participacoes_por_prof = defaultdict(list)
premios_por_prof = defaultdict(list)
for p in participacoes:
    participacoes_por_prof[p['chave']].append(p)
for premio in premios_unicos.values():
    premios_por_prof[premio['chave']].append(premio)

COLUNAS = [
    'Professor(a)', 'Situação no cadastro', 'Gênero', 'Raça/cor',
    'Orientador', 'Co-orientador', 'Banca titular', 'Banca suplente',
    'Bancas distintas', 'Total de funções', 'Quantidade de prêmios/menções',
    'Quais prêmios/menções', 'Finalistas/indicações', 'Outros reconhecimentos',
    'Anos de defesas', 'Vínculos inferidos', 'Observações'
]

resumo = []
for cod, pessoa in sorted(pessoas.items(), key=lambda par: normalizar(par[1]['nome'])):
    linhas = participacoes_por_prof[cod]
    premios = premios_por_prof[cod]
    contagem = Counter(p['funcao'] for p in linhas)
    principais = [a for a in premios if a['tipo']=='Prêmio/menção informada']
    bancas_distintas = len({p['linha_excel'] for p in linhas})
    inferidos = sum('inferido' in p['criterio_vinculo'].lower()
                    or 'sem unidade' in p['criterio_vinculo'].lower() for p in linhas)
    observacoes = []
    if cod not in cadastro:
        observacoes.append('Fora do cadastro nominal de gênero e raça')
    if not linhas:
        observacoes.append('Sem participação identificada em 2023–2025')
    if inferidos:
        observacoes.append('Há participações com vínculo IC inferido')
    resumo.append({
        'Professor(a)': pessoa['nome'],
        'Situação no cadastro': pessoa['situacao'],
        'Gênero': pessoa['genero'],
        'Raça/cor': pessoa['raca_cor'],
        'Orientador': contagem['Orientador'],
        'Co-orientador': contagem['Co-orientador'],
        'Banca titular': sum(n for funcao,n in contagem.items() if funcao.startswith('Titular')),
        'Banca suplente': sum(n for funcao,n in contagem.items() if funcao.startswith('Suplente')),
        'Bancas distintas': bancas_distintas,
        'Total de funções': len(linhas),
        'Quantidade de prêmios/menções': len(principais),
        'Quais prêmios/menções': ' | '.join((str(a['ano'])+' — ' if a['ano'] else '') + a['titulo'] for a in principais),
        'Finalistas/indicações': sum(a['tipo']=='Finalista/indicação' for a in premios),
        'Outros reconhecimentos': sum(a['tipo']=='Outros reconhecimentos' for a in premios),
        'Anos de defesas': ', '.join(str(a) for a in sorted({p['ano'] for p in linhas})),
        'Vínculos inferidos': inferidos,
        'Observações': '; '.join(observacoes),
    })

print('Pessoas na tabela:', len(resumo))
print('Total de orientações:', sum(p['Orientador'] for p in resumo))
print('Total de coorientações:', sum(p['Co-orientador'] for p in resumo))
print('Total de participações como titular:', sum(p['Banca titular'] for p in resumo))
print('Total de participações como suplente:', sum(p['Banca suplente'] for p in resumo))

Pessoas na tabela: 95
Total de orientações: 173
Total de coorientações: 53
Total de participações como titular: 346
Total de participações como suplente: 153


In [13]:
# Salvando o resultado
with open(saida('tabela_docentes_ic.csv'), 'w', encoding='utf-8-sig', newline='') as arq:
    escritor = csv.DictWriter(arq, fieldnames=COLUNAS, delimiter=';')
    escritor.writeheader()
    escritor.writerows(resumo)
print('CSV salvo em: tabela_docentes_ic.csv')

try:
    from artifact_tool import Workbook, SpreadsheetFile

    wb = Workbook.create()
    def aba(nome, titulos, linhas, larguras=None):
        s = wb.worksheets.add(nome)
        if linhas:
            s.get_range_by_indexes(0,0,len(linhas)+1,len(titulos)).values = [titulos] + linhas
        else:
            s.get_range_by_indexes(0,0,1,len(titulos)).values = [titulos]
        cab = s.get_range_by_indexes(0,0,1,len(titulos))
        cab.format = {'fill':'#155E75','font':{'bold':True,'color':'#FFFFFF','size':10},
                      'row_height':35,'wrap_text':True}
        s.freeze_panes.freeze_rows(1)
        if linhas:
            s.get_range_by_indexes(1,0,len(linhas),len(titulos)).format.row_height = 21
        if larguras:
            for j,w in enumerate(larguras):
                s.get_range_by_indexes(0,j,len(linhas)+1,1).format.column_width = w
        return s

    aba('Resumo', COLUNAS, [[linha[c] for c in COLUNAS] for linha in resumo],
        [39,32,17,18,15,17,17,18,17,18,25,80,21,23,21,20,58])

    DETALHE = ['Professor(a)','Ano','Nível','Aluno(a)','Função','Instituto','Tipo de vínculo','Linha da unificada']
    aba('Participações', DETALHE,
        [[p[c] for c in ('nome','ano','nivel','aluno','funcao','instituto','criterio_vinculo','linha_excel')]
         for p in participacoes], [39,12,17,43,18,22,41,24])

    PREMIOS_COL = ['Professor(a)','Ano prêmio','Título','Categoria','Situação da premiação','Linhas de participação na unificada']
    aba('Prêmios', PREMIOS_COL,
        [[p['nome'],p['ano'],p['titulo'],p['categoria'],p['situacao'],', '.join(map(str,sorted(p['linhas_fonte'])))]
         for p in premios_unicos.values()], [39,17,86,40,30,40])

    aba('Pendências', ['Ano','Aluno(a)','Nome informado','Função','Instituto','Linha','Motivo'],
        pendencias, [14,42,44,21,25,14,72])

    NOTAS = [
        ['Período das participações','Defesas de mestrado e doutorado 2023–2025 da classificação unificada'],
        ['Gênero e raça','Somente valores transcritos da relação de gênero e raça do IC; sem inferência'],
        ['Vínculo IC','Explícito quando instituto IC/UNICAMP; algumas ausências incluídas por cadastro/ocorrência anterior'],
        ['Bancas distintas','Cada combinação professor e linha de defesa contada uma vez, independentemente de múltiplos papéis'],
        ['Prêmios','Fonte única: classificacao_detalhada.csv; não depende de participação em banca'],
        ['Deduplicação dos prêmios','Base oficial já deduplicada; chave validada sem excluir registros'],
        ['Finalistas/outros reconhecimentos','Contados à parte e não somados a prêmios/menções'],
        ['Ausência de registro','Zero não significa ausência histórica de participação ou prêmio'],
        ['Ano de prêmio','Ano da base oficial; período completo, inclusive prêmios posteriores à defesa'],
        ['Validação','Conferir as abas Pendências, Participações e Prêmios antes de publicar os dados'],
    ]
    aba('Metodologia', ['Regra','Explicação'], NOTAS, [35,110])

    SpreadsheetFile.export_xlsx(wb).save(str(saida('tabela_docentes_ic_com_codigo.xlsx')))
    print('Excel salvo em: tabela_docentes_ic_com_codigo.xlsx')
except Exception as erro:
    print(f"O CSV foi salvo; Excel opcional não gerado: {erro}")

assert sum(r['Total de funções'] for r in resumo) == len(participacoes)
assert sum(r['Quantidade de prêmios/menções'] for r in resumo) == sum(
    p['tipo'] == 'Prêmio/menção informada' for p in premios_unicos.values())
assert len(resumo) == len(pessoas)
print('Verificações passaram!')
print('Resumo: pessoas=', len(resumo), 'defesas=', len(defesa_validas),
      'participações=', len(participacoes),
      'prêmios/menções=', sum(p['Quantidade de prêmios/menções'] for p in resumo))
print('Exemplo (primeiros 3 docentes):')
for r in resumo[:3]:
    print(r['Professor(a)'], ':', r['Orientador'], 'orient.,',
          r['Co-orientador'], 'coorient.,', r['Quantidade de prêmios/menções'], 'prêmios')

CSV salvo em: tabela_docentes_ic.csv
O CSV foi salvo; Excel opcional não gerado: No module named 'artifact_tool'
Verificações passaram!
Resumo: pessoas= 95 defesas= 173 participações= 725 prêmios/menções= 249
Exemplo (primeiros 3 docentes):
Alexandre Mello Ferreira : 0 orient., 1 coorient., 0 prêmios
Alexandre Xavier Falcão : 11 orient., 0 coorient., 14 prêmios
Allan Mariano de Souza : 1 orient., 1 coorient., 3 prêmios


In [14]:
tabela_docentes = pd.read_csv(saida('tabela_docentes_ic.csv'), sep=";")
print(tabela_docentes)
tabela_docentes = tabela_docentes[
    tabela_docentes["Situação no cadastro"] != "Verificar vínculo"
].copy()

# Reorganizar os índices
tabela_docentes.reset_index(drop=True, inplace=True)

display(tabela_docentes)
print("Total de professores:", len(tabela_docentes))
print(tabela_docentes["Situação no cadastro"].value_counts())

# Validação da coerência inclusive após o filtro final, por docente e situação.
assert tabela_docentes['Professor(a)'].map(chave).is_unique
por_nome = tabela_docentes.set_index(tabela_docentes['Professor(a)'].map(chave))
campos = {'Prêmio/menção informada': 'Quantidade de prêmios/menções',
          'Finalista/indicação': 'Finalistas/indicações',
          'Outros reconhecimentos': 'Outros reconhecimentos'}
for cod, linha in por_nome.iterrows():
    esperado = Counter(p['tipo'] for p in premios_unicos.values() if p['chave'] == cod)
    for tipo, coluna in campos.items():
        assert int(linha[coluna]) == esperado[tipo], (cod, coluna)
assert {p['chave'] for p in premios_unicos.values()}.issubset(por_nome.index)
assert sum(int(tabela_docentes[c].sum()) for c in campos.values()) == len(base_premios)
print('Coerência validada: todos os registros oficiais presentes no CSV filtrado.')
print(tabela_docentes[list(campos.values())].sum().to_string())

tabela_docentes.to_csv(
    saida("tabela_docentes_ic_filtrada.csv"),
    sep=";",
    index=False,
    encoding="utf-8-sig"
)

                        Professor(a) Situação no cadastro         Gênero  \
0           Alexandre Mello Ferreira    Verificar vínculo  Não informado   
1            Alexandre Xavier Falcão               Ativos      Homem cis   
2             Allan Mariano de Souza               Ativos      Homem cis   
3                    Anamaria Gomide          Aposentados     Mulher cis   
4            Anderson Anjos da Silva    Verificar vínculo  Não informado   
..                               ...                  ...            ...   
90           Siome Klein Goldenstein       Ex-professores      Homem cis   
91            Thelma Cecília Chiossi          Aposentados     Mulher cis   
92                Tomasz Kowaltowski          Aposentados      Homem cis   
93  Washington Luiz Miranda da Cunha               Ativos      Homem cis   
94                       Zanoni Dias               Ativos      Homem cis   

         Raça/cor  Orientador  Co-orientador  Banca titular  Banca suplente  \
0   Não 

,Professor(a),Situação no cadastro,Gênero,Raça/cor,Orientador,Co-orientador,Banca titular,Banca suplente,Bancas distintas,Total de funções,Quantidade de prêmios/menções,Quais prêmios/menções,Finalistas/indicações,Outros reconhecimentos,Anos de defesas,Vínculos inferidos,Observações
0,Alexandre Xavier Falcão,Ativos,Homem cis,Branco,11,0,11,1,12,23,14,2019 — Melhor artigo (voto popular): ALTIS: A ...,0,2,"2023, 2024, 2025",1,Há participações com vínculo IC inferido
1,Allan Mariano de Souza,Ativos,Homem cis,Branco,1,1,4,6,11,12,3,2023 — Best Paper Award XLI Simpósio Brasileir...,0,0,"2023, 2024, 2025",1,Há participações com vínculo IC inferido
2,Anamaria Gomide,Aposentados,Mulher cis,Branco,0,0,0,0,0,0,0,NaN,0,0,NaN,0,Sem participação identificada em 2023–2025
3,Anderson de Rezende Rocha,Ativos,Homem cis,Branco,9,1,11,6,18,27,15,2019 — Google Research Awards for Latin Americ...,0,12,"2024, 2025",1,Há participações com vínculo IC inferido
4,Andre Santanche,Ativos,Homem cis,Branco,4,1,14,9,24,28,4,2021 — Prêmio Menção Honrosa Artigo Curto no S...,0,0,"2023, 2024, 2025",1,Há participações com vínculo IC inferido
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
80,Siome Klein Goldenstein,Ex-professores,Homem cis,Branco,0,0,0,0,0,0,0,NaN,0,0,NaN,0,Sem participação identificada em 2023–2025
81,Thelma Cecília Chiossi,Aposentados,Mulher cis,Branco,0,0,0,0,0,0,0,NaN,0,0,NaN,0,Sem participação identificada em 2023–2025
82,Tomasz Kowaltowski,Aposentados,Homem cis,Branco,0,0,0,0,0,0,0,NaN,0,0,NaN,0,Sem participação identificada em 2023–2025
83,Washington Luiz Miranda da Cunha,Ativos,Homem cis,Pardo?,0,0,0,0,0,0,0,NaN,0,0,NaN,0,Sem participação identificada em 2023–2025


Total de professores: 85
Situação no cadastro
Ativos            47
Aposentados       25
Ex-professores     6
Colaboradores      4
In Memoriam        3
Name: count, dtype: int64
Coerência validada: todos os registros oficiais presentes no CSV filtrado.
Quantidade de prêmios/menções    249
Finalistas/indicações             16
Outros reconhecimentos            28
